# FairGov AI DTE — Google Colab ML Evaluation
This notebook runs the existing GitHub ML source in Colab. Synthetic records only; not real scholarship outcomes. Sequence: dataset generation → validation → XGBoost → evaluation → SHAP → boundary audit.

In [ ]:
%pip -q install "xgboost>=2,<4" "shap>=0.46,<1" "scikit-learn>=1.4,<2" pandas numpy matplotlib joblib fastapi pydantic pytest

In [ ]:
!git clone -q https://github.com/Prashantpowar45/FairGov---AI.git
%cd FairGov---AI

## 1. Generate, validate and train

In [ ]:
!python -m src.pipeline

## 2. Dataset quality and metrics

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
display(pd.DataFrame([json.loads(Path('reports/dataset_validation.json').read_text())]))
display(pd.DataFrame([json.loads(Path('reports/model_evaluation.json').read_text())]))
df=pd.read_csv('data/synthetic_applicants.csv')
display(df.head())
print('Class counts:',df.eligible.value_counts().to_dict())

## 3. Model feature importance and SHAP

In [ ]:
import joblib
import matplotlib.pyplot as plt
from src.rules import FEATURES
import shap
model=joblib.load('models/xgboost_dte.joblib')
X=df[FEATURES]
imp=pd.Series(model.feature_importances_,index=FEATURES).sort_values()
imp.plot.barh(figsize=(7,5),title='Synthetic-model XGBoost feature importance');plt.tight_layout();plt.show()
example=X.iloc[[0]]
shap_exp=shap.TreeExplainer(model)(example)
shap.plots.waterfall(shap_exp[0],show=True)

## 4. Independent manually expected boundary test cases
The expected outcomes below are manual *demo rule* expectations. These are not actual government approvals.

In [ ]:
from src.rules import check
base={'annual_income':600000,'maharashtra_domicile':1,'indian_national':1,'cap_admission':1,'bonafide':1,'professional_course':1,'private_or_deemed_university':0,'other_scholarship':0,'family_beneficiaries':1,'attendance_pct':75,'first_semester':0,'gap_years':0}
cases=[('income limit',{'annual_income':800000},True),('income too high',{'annual_income':800001},False),('attendance 50%',{'attendance_pct':50},True),('attendance 49%',{'attendance_pct':49},False),('first semester exception',{'attendance_pct':49,'first_semester':1},True),('two year gap',{'gap_years':2},False),('missing CAP',{'cap_admission':0},False)]
rows=[]
for name,change,expected in cases:
    student={**base,**change}
    rule=check(student)['preliminary_eligible']
    ml=bool(model.predict(pd.DataFrame([student])[FEATURES])[0])
    rows.append({'case':name,'expected_rule_result':expected,'actual_rule_result':rule,'ml_prediction':ml,'mismatch':ml!=rule,'boundary_rule_pass':rule==expected})
audit=pd.DataFrame(rows);display(audit)
assert audit.boundary_rule_pass.all()
audit.to_csv('fairgov_boundary_audit.csv',index=False)

## 5. Run tests and export evidence

In [ ]:
!python -m pytest -q

**Limitations:** Rules and labels use a simplified scheme subset. Validation here measures internal rule consistency and manually specified demo boundary cases. ML performance on synthetic data does not establish real-government accuracy. This is an ML-module prototype, not independently verified 50% full-system completion.